In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import matplotlib.colors as mcolors
from matplotlib.cm import ScalarMappable
from matplotlib.colors import ListedColormap
import geopandas as gpd

In [4]:
reference_file = "//sce.eix.com/workgroup/SPE_DSA/Charles/IgnitionGoalSettingInput/Distribution_2015-2024_WithCircuitLengths.csv"
reference_df = pd.read_csv(reference_file, low_memory=False, nrows=10)

In [5]:
reference_df

,time,Circuit_Name_FuelData,max_max_fpi_2.0_tdd_fuels_index,mean_max_fpi_2.0_tdd_fuels_index,median_max_fpi_2.0_tdd_fuels_index,p75_max_fpi_2.0_tdd_fuels_index,p90_max_fpi_2.0_tdd_fuels_index,p95_max_fpi_2.0_tdd_fuels_index,max_mean_fpi_2.0_tdd_fuels_index,mean_mean_fpi_2.0_tdd_fuels_index,...,mean_p95_lfpw_using_dew_point_depression,median_p95_lfpw_using_dew_point_depression,p75_p95_lfpw_using_dew_point_depression,p90_p95_lfpw_using_dew_point_depression,p95_p95_lfpw_using_dew_point_depression,Date,Year,Circuit Name,Circuit FLOC,Circuit Miles
0,2015-01-01T00:00:00-08:00,HI LINE,0.287345,0.265334,0.265383,0.282467,0.287048,0.287233,0.192295,0.172548,...,0.959767,0.663448,1.358356,2.001346,2.175403,2015-01-01,2015,HI LINE,ED-08427,25.1811
1,2015-01-02T00:00:00-08:00,HI LINE,0.298353,0.274315,0.271084,0.286612,0.293096,0.294020,0.186456,0.165580,...,0.524927,0.526151,0.637662,0.713337,0.724252,2015-01-02,2015,HI LINE,ED-08427,25.1811
2,2015-01-03T00:00:00-08:00,HI LINE,0.290336,0.267696,0.261288,0.276733,0.287329,0.287826,0.172443,0.156317,...,0.193335,0.197823,0.211645,0.283680,0.372936,2015-01-03,2015,HI LINE,ED-08427,25.1811
3,2015-01-04T00:00:00-08:00,HI LINE,0.288494,0.265099,0.266496,0.272691,0.286355,0.287156,0.159368,0.146211,...,0.829329,0.401402,1.031231,2.363874,2.576631,2015-01-04,2015,HI LINE,ED-08427,25.1811
4,2015-01-05T00:00:00-08:00,HI LINE,0.331302,0.306010,0.312253,0.319900,0.329338,0.330632,0.183952,0.169220,...,2.433482,2.552127,2.876201,3.339849,3.381666,2015-01-05,2015,HI LINE,ED-08427,25.1811
5,2015-01-06T00:00:00-08:00,HI LINE,0.361078,0.332225,0.332956,0.347340,0.356769,0.359333,0.201004,0.191008,...,2.894470,3.337116,3.933100,4.634165,4.842501,2015-01-06,2015,HI LINE,ED-08427,25.1811
6,2015-01-07T00:00:00-08:00,HI LINE,0.358322,0.335656,0.332532,0.342822,0.352999,0.354973,0.200161,0.187164,...,1.946140,1.313471,2.977721,3.964264,5.699205,2015-01-07,2015,HI LINE,ED-08427,25.1811
7,2015-01-08T00:00:00-08:00,HI LINE,0.328490,0.307130,0.309880,0.322091,0.325311,0.327002,0.183475,0.160184,...,0.448472,0.226995,0.767731,0.958761,0.960948,2015-01-08,2015,HI LINE,ED-08427,25.1811
8,2015-01-09T00:00:00-08:00,HI LINE,0.273774,0.226315,0.221117,0.239519,0.257177,0.260529,0.146378,0.114675,...,0.167746,0.122171,0.283305,0.323863,0.352350,2015-01-09,2015,HI LINE,ED-08427,25.1811
9,2015-01-10T00:00:00-08:00,HI LINE,0.188444,0.149259,0.144366,0.158240,0.170700,0.179183,0.102081,0.078210,...,0.297314,0.252711,0.437679,0.544229,0.572028,2015-01-10,2015,HI LINE,ED-08427,25.1811


# Share 2025 Historical Data to Adrienne

In [6]:
# read circuit lengths
circuitlengths = pd.read_excel("//sce.eix.com/workgroup/SPE_DSA/Charles/IgnitionGoalSettingInput/Q1_2025-_HFRA_Conductor_SumStat_Circuit.xlsx")
# filter for OH only
circuitlengths = circuitlengths[circuitlengths['Conductor Type'].str.contains('OH',na=False)]
# filter for HFRA only
circuitlengths = circuitlengths[circuitlengths['HFRA Tier'].notna()]
# remove blank circuits
circuitlengths = circuitlengths[circuitlengths['Circuit Name'].notna()]
# capitalize circuit names
circuitlengths['Circuit Name'] = circuitlengths['Circuit Name'].str.upper()
circuitlengths_map = circuitlengths.groupby(['Circuit Name','Circuit FLOC'])['Circuit Miles'].sum().reset_index()

In [7]:
# read 2025 historical data for the first half
historical_2025_first_half = pd.read_csv(r"\\sce.eix.com\workgroup\SPE_DSA\Charles\IgnitionGoalSettingMidput\Historical_Fuel_07012025.csv", low_memory=False)
historical_2025_first_half['Date'] = pd.to_datetime(historical_2025_first_half['time']).dt.strftime('%Y-%m-%d')
historical_2025_first_half['Year'] = pd.to_datetime(historical_2025_first_half['time']).dt.year
# rename column id to Circuit_Name_FuelData
historical_2025_first_half = historical_2025_first_half.rename(columns={'id': 'Circuit_Name_FuelData'})
# merge with circuit lengths to get circuit miles
historical_2025_first_half = pd.merge(historical_2025_first_half, circuitlengths_map, how='left', left_on='Circuit_Name_FuelData', right_on='Circuit Name')
historical_2025_first_half

,time,Circuit_Name_FuelData,max_max_fpi_2.0_tdd_fuels_index,min_max_fpi_2.0_tdd_fuels_index,median_max_fpi_2.0_tdd_fuels_index,mean_max_fpi_2.0_tdd_fuels_index,max_min_fpi_2.0_tdd_fuels_index,min_min_fpi_2.0_tdd_fuels_index,median_min_fpi_2.0_tdd_fuels_index,mean_min_fpi_2.0_tdd_fuels_index,...,mean_median_fpi_2.0_tdd_fuels_index,max_mean_fpi_2.0_tdd_fuels_index,min_mean_fpi_2.0_tdd_fuels_index,median_mean_fpi_2.0_tdd_fuels_index,mean_mean_fpi_2.0_tdd_fuels_index,Date,Year,Circuit Name,Circuit FLOC,Circuit Miles
0,2025-01-01T00:00:00-08:00,HI LINE,0.698310,0.629074,0.674154,0.666318,0.340334,0.229061,0.316036,0.296836,...,0.473728,0.512060,0.419408,0.485415,0.473152,2025-01-01,2025,HI LINE,ED-08427,25.181100
1,2025-01-02T00:00:00-08:00,HI LINE,0.762892,0.686161,0.742840,0.731656,0.428417,0.339642,0.408212,0.390390,...,0.536991,0.585352,0.503839,0.567722,0.552305,2025-01-02,2025,HI LINE,ED-08427,25.181100
2,2025-01-03T00:00:00-08:00,HI LINE,0.786906,0.701640,0.753019,0.750322,0.440116,0.387112,0.428559,0.421836,...,0.554965,0.609967,0.553240,0.571729,0.578428,2025-01-03,2025,HI LINE,ED-08427,25.181100
3,2025-01-04T00:00:00-08:00,HI LINE,0.764035,0.673567,0.720486,0.725014,0.430499,0.350454,0.384550,0.387266,...,0.522711,0.582374,0.506393,0.531874,0.541472,2025-01-04,2025,HI LINE,ED-08427,25.181100
4,2025-01-05T00:00:00-08:00,HI LINE,0.807205,0.725945,0.784558,0.775080,0.443786,0.364163,0.421819,0.409951,...,0.566568,0.621277,0.535579,0.598487,0.586500,2025-01-05,2025,HI LINE,ED-08427,25.181100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
676030,2025-06-26T00:00:00-08:00,TRUMPET,0.901151,0.825393,0.876081,0.868858,0.823536,0.746576,0.799008,0.788902,...,0.848155,0.873180,0.796597,0.846811,0.839710,2025-06-26,2025,TRUMPET,ED-18234,41.115027
676031,2025-06-27T00:00:00-08:00,TRUMPET,0.914479,0.855988,0.887854,0.887664,0.841140,0.775378,0.810529,0.811016,...,0.867839,0.887190,0.823919,0.859629,0.858702,2025-06-27,2025,TRUMPET,ED-18234,41.115027
676032,2025-06-28T00:00:00-08:00,TRUMPET,0.923443,0.871935,0.902022,0.900223,0.850503,0.797779,0.826139,0.826110,...,0.879769,0.895856,0.843128,0.873734,0.871932,2025-06-28,2025,TRUMPET,ED-18234,41.115027
676033,2025-06-29T00:00:00-08:00,TRUMPET,0.938028,0.892641,0.920970,0.917982,0.868339,0.824579,0.848068,0.847692,...,0.898248,0.911142,0.863571,0.891967,0.889559,2025-06-29,2025,TRUMPET,ED-18234,41.115027


In [8]:
# save a file for Adrienne
historical_2025_first_half.to_csv("//sce.eix.com/workgroup/SPE_DSA/Charles/IgnitionGoalSettingInput/Historical_2025_First_Half_WithCircuitLengths.csv", index=False)

In [9]:
historical_2025_first_half['Category'] = np.where(historical_2025_first_half['mean_max_fpi_2.0_tdd_fuels_index'] >= 0.85, "High", "Low")
historical_2025_first_half[historical_2025_first_half['Category'] == "High"]['Circuit Miles'].sum()

np.float64(157708.05510612112)